# Task 1 — Import the data & examine variables

**Problem-statement step:** *After importing the data, examine variables such as `Dt_Customer` and `Income` to verify their accurate importation.*

We load the raw file, expose the two columns pandas imports incorrectly, fix them, and prove the fix worked.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")

# Locate the raw data whether the notebook is opened from its own folder
# (notebooks/) or from the project root.
import os
_CANDIDATES = ["marketing_data.csv", "../marketing_data.csv",
               os.path.join(os.path.dirname(os.getcwd()), "marketing_data.csv")]
DATA_PATH = next((p for p in _CANDIDATES if os.path.exists(p)), "marketing_data.csv")
print("Using data file:", DATA_PATH)

Using data file: ../marketing_data.csv


## 1. Load the raw data exactly as delivered

In [2]:
raw = pd.read_csv(DATA_PATH)
print("Shape:", raw.shape)
raw.head()

Shape: (2240, 28)


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Response,Complain,Country
0,1826,1970,Graduation,Divorced,"$84,835.00",0,0,6/16/14,0,189,104,379,111,189,218,1,4,4,6,1,0,0,0,0,0,1,0,SP
1,1,1961,Graduation,Single,"$57,091.00",0,0,6/15/14,0,464,5,64,7,0,37,1,7,3,7,5,0,0,0,0,1,1,0,CA
2,10476,1958,Graduation,Married,"$67,267.00",0,1,5/13/14,0,134,11,59,15,2,30,1,3,2,5,2,0,0,0,0,0,0,0,US
3,1386,1967,Graduation,Together,"$32,474.00",1,1,5/11/14,0,10,0,1,0,0,0,1,1,0,2,7,0,0,0,0,0,0,0,AUS
4,5371,1989,Graduation,Single,"$21,474.00",1,0,4/8/14,0,6,16,24,11,0,34,2,3,1,2,7,1,0,0,0,0,1,0,SP


## 2. Inspect the raw column names and dtypes
Note the header `' Income '` carries **stray spaces**, and both `Income` and `Dt_Customer` come in as text.

In [3]:
print(list(raw.columns))

['ID', 'Year_Birth', 'Education', 'Marital_Status', ' Income ', 'Kidhome', 'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1', 'AcceptedCmp2', 'Response', 'Complain', 'Country']


In [4]:
raw.dtypes

ID                     int64
Year_Birth             int64
Education                str
Marital_Status           str
 Income                  str
Kidhome                int64
Teenhome               int64
Dt_Customer              str
Recency                int64
MntWines               int64
MntFruits              int64
MntMeatProducts        int64
MntFishProducts        int64
MntSweetProducts       int64
MntGoldProds           int64
NumDealsPurchases      int64
NumWebPurchases        int64
NumCatalogPurchases    int64
NumStorePurchases      int64
NumWebVisitsMonth      int64
AcceptedCmp3           int64
AcceptedCmp4           int64
AcceptedCmp5           int64
AcceptedCmp1           int64
AcceptedCmp2           int64
Response               int64
Complain               int64
Country                  str
dtype: object

## 3. Examine the two suspicious variables *before* fixing

In [5]:
income_col = " Income "  # the real header, with spaces
print("Income samples :", raw[income_col].head(3).tolist(), "| dtype:", raw[income_col].dtype)
print("Dt samples     :", raw["Dt_Customer"].head(3).tolist(), "| dtype:", raw["Dt_Customer"].dtype)
print("Missing Income :", int(raw[income_col].isna().sum()))

Income samples : ['$84,835.00 ', '$57,091.00 ', '$67,267.00 '] | dtype: str
Dt samples     : ['6/16/14', '6/15/14', '5/13/14'] | dtype: str
Missing Income : 24


## 4. Fix the data types
* strip whitespace from every column name,
* convert `Income` from a currency string to `float`,
* parse `Dt_Customer` (format `m/d/yy`) to a real datetime.

In [6]:
df = raw.copy()
df.columns = df.columns.str.strip()
df["Income"] = (df["Income"].astype("string")
                .str.replace(r"[\$,]", "", regex=True).str.strip().astype(float))
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%m/%d/%y")
df[["Income", "Dt_Customer"]].dtypes

Income                float64
Dt_Customer    datetime64[us]
dtype: object

## 5. Examine the corrected variables *after* fixing

In [7]:
print("Income dtype :", df["Income"].dtype, "| sample:", df["Income"].head(3).tolist())
print("Dt range     :", df["Dt_Customer"].min().date(), "->", df["Dt_Customer"].max().date())
df["Income"].describe().round(2)

Income dtype : float64 | sample: [84835.0, 57091.0, 67267.0]
Dt range     : 2012-07-30 -> 2014-06-29


count      2216.00
mean      52247.25
std       25173.08
min        1730.00
25%       35303.00
50%       51381.50
75%       68522.00
max      666666.00
Name: Income, dtype: float64

### Conclusion
* `Income` header had surrounding whitespace and currency-string values — now `float64`.
* `Dt_Customer` was text — now `datetime64` spanning 2012–2014.
* 24 `Income` values are missing → handled in **Task 2**.